# 02 — Address Anchor Blocking
**Amazon ML Challenge 2026: Business Entity Resolution**

Evaluates structural address blocking keys (PIN, PIN+Primary Number, PIN+Token) and bucket distributions.


### 1. Environment & Setup


In [ ]:
import os
import sys
import pandas as pd
import numpy as np

# Adjust sys.path to root directory
if '..' not in sys.path:
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../..')))
if '.' not in sys.path:
    sys.path.insert(0, os.getcwd())

from src.business_entity_resolution.blocking import (
    build_or_load_benchmark,
    evaluate_candidate_pairs,
    format_evaluation_summary,
    CandidateUnionEngine
)


### 2. Load Benchmark Dataset


In [ ]:
s1, s2, s3, gt = build_or_load_benchmark(n_s1=5000, n_distractors=100000, random_seed=42)
target_pool = pd.concat([s2, s3], ignore_index=True)


### 3. Run Address Anchor Blocker


In [ ]:
from src.business_entity_resolution.blocking import AddressBlockConfig, AddressAnchorBlocker

cfg = AddressBlockConfig(use_pin_only=False, use_pin_primary_num=True, use_pin_name_token=True, max_bucket_size=500)
blk = AddressAnchorBlocker(name="Block_B", config=cfg)
blk.fit(target_pool)
cands = blk.block(s1)

metrics = evaluate_candidate_pairs(cands, gt, len(target_pool))
print(format_evaluation_summary(metrics, title="Address Anchor Blocker"))
print("Bucket Diagnostics:", blk.get_bucket_diagnostics())
